# EXP2: Chord Quality Classification

## Centralized imports
Imports all required libraries. Run this cell first.

In [ ]:
# Centralized imports

import os
import sys
import json
import glob
import shutil
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
    PrinterCallback,
)
from transformers.utils import logging as hf_logging

import safetensors.torch as st

from IPython.display import display, Image as IPImage



## Cell 0 — Path Configuration
Central path registry for the entire notebook.
Every downstream cell reads paths from the `paths` object defined here — no hardcoded
directories anywhere else. Update **only this cell** when migrating between machines.

In [ ]:
# Cell 0 — Path Configuration

@dataclass
class ExpPaths:
    base:       str = os.getenv("SECD_BASE", str(Path.cwd()))
    cache_root: str = ""
    outdir:     str = ""
    split_dir:  str = ""
    csv_dirs:   list = field(default_factory=list)

    def __post_init__(self):
        self.cache_root = os.path.join(self.base, "mel_cache_ast16k_256")
        self.outdir     = os.path.join(self.base, "EXP2_AST_TRIADS")
        self.split_dir  = os.path.join(self.outdir, "exp2_splits")

        self.csv_dirs = []
        for group in [
            "triads_loose", "triads_strict",
        ]:
            d = os.path.join(self.base, group, "csv")
            if os.path.isdir(d):
                self.csv_dirs.append(d)

    def verify(self):
        errors = []
        if not os.path.isdir(self.cache_root):
            errors.append(f"Cache not found: {self.cache_root}")
        if not self.csv_dirs:
            errors.append("No CSV directories found")
        for d in self.csv_dirs:
            csvs = [f for f in os.listdir(d)
                    if f.endswith(".csv") and "-original" not in f]
            if not csvs:
                errors.append(f"No usable CSVs in {d}")
        if errors:
            for e in errors:
                print(f"  ❌ {e}")
            raise RuntimeError("Path verification failed")

        os.makedirs(self.outdir, exist_ok=True)
        os.makedirs(self.split_dir, exist_ok=True)

        all_csvs = []
        for d in self.csv_dirs:
            all_csvs += [os.path.join(d, f) for f in os.listdir(d)
                         if f.endswith(".csv") and "-original" not in f]

        print(f"  ✅ Base       : {self.base}")
        print(f"  ✅ Cache      : {self.cache_root}")
        print(f"  ✅ Output     : {self.outdir}")
        print(f"  ✅ Splits     : {self.split_dir}")
        print(f"  ✅ CSV dirs   : {len(self.csv_dirs)}")
        print(f"  ✅ CSV files  : {len(all_csvs)} (excluding -original)")
        return all_csvs

paths = ExpPaths()
all_csv_paths = paths.verify()
print(f"\n✅ Path configuration OK")


## Cell 3 — Constants, Paths, CSV Inventory (EXP2: triad chord-type)
Establishes all experiment-level constants for EXP2.
EXP2 classifies **global triad chord type** (major / minor / diminished / augmented).
Uses `triads_loose` and `triads_strict` subsets as the two metadata sources for this task.


In [ ]:
# Cell 3 — Constants, paths, CSV inventory (EXP2: triad chord-type)

# ===================== Instruments =====================
INSTRUMENTS = ["cello", "viola", "violin2", "violin1"]
INSTR_ORDER  = INSTRUMENTS
assert INSTR_ORDER == ["cello", "viola", "violin2", "violin1"]

# ===================== Audio / Feature constants =====================
TARGET_SR     = 16000   # AST-native sample rate
TARGET_FRAMES = 256     # ~2.6 sec at 16 kHz / 10 ms hop (fits 2-sec clips)
N_MELS        = 128
DURATION      = 2.0     # clip length in seconds

# ===================== Reproducibility =====================
SEED = 1337

# ===================== Model =====================
MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
TASK_NAME = "EXP2 triad chord-type classification"
TASK_OBJECTIVE = "Global triad chord-quality classification from precomputed mel-spectrogram inputs."
IGNORE_INDEX = -1
LABEL_SMOOTHING = 0.05

# ===================== Filesystem (from Cell 0) =====================
CACHE_ROOT = paths.cache_root
os.makedirs(paths.outdir, exist_ok=True)

# ===================== Per-instrument column helpers =====================
def _cand_dyn_cols(inst: str):
    return [f"{inst}_dynamic", f"{inst}_dyn", f"dynamic_{inst}", f"dyn_{inst}"]

def _cand_tec_cols(inst: str):
    return [f"{inst}_technique", f"{inst}_tec", f"technique_{inst}", f"tec_{inst}"]

# ===================== CSV inventory (triads only) =====================
_CSV_MAP = {
    "trios_loose": ("triads_loose", [
        "triads_major_loose.csv", "triads_minor_loose.csv",
        "triads_diminished_loose.csv", "triads_augmented_loose.csv"]),
    "trios_strict": ("triads_strict", [
        "triads_major_strict.csv", "triads_minor_strict.csv",
        "triads_diminished_strict.csv", "triads_augmented_strict.csv"]),
}

def _build_csv_list(group_key):
    folder, files = _CSV_MAP[group_key]
    return [os.path.join(paths.base, folder, "csv", f) for f in files]

csv_trios_loose     = _build_csv_list("trios_loose")
csv_trios_strict    = _build_csv_list("trios_strict")

ALL_CSVS = csv_trios_loose + csv_trios_strict

# ===================== Fail-fast inventory check =====================
_missing = [p for p in ALL_CSVS if not os.path.isfile(p)]
if _missing:
    print("❌ Missing CSV files:")
    for p in _missing: print("  -", p)
    raise FileNotFoundError(f"{len(_missing)} CSV(s) missing — fix paths and re-run this cell.")

print(f"   CSV inventory OK — {len(ALL_CSVS)} files found.")
print(f"   CACHE_ROOT : {CACHE_ROOT}")
print(f"   OUTDIR     : {paths.outdir}")


## Cell 4 — Load Metadata & Derive Chord-Type Label (EXP2)
Loads all triad CSVs and derives the **chord_type** label from the CSV filename
(e.g. `triads_major_loose.csv` → `major`). This is the global classification target for EXP2.

In [ ]:
# Cell 4 — Load metadata & derive chord-type label (EXP2)

CANDIDATE_FILENAME_COLS = ["chord_filename", "filename", "wav_filename", "output_wav", "wav_file"]

def resolve_filename_column(df: pd.DataFrame) -> str:
    for c in CANDIDATE_FILENAME_COLS:
        if c in df.columns:
            return c
    raise KeyError(f"Filename column not found in CSV. Tried: {CANDIDATE_FILENAME_COLS}")

def derive_audio_root_from_csv(csv_path: str) -> str:
    csv_dir, csv_file = os.path.split(csv_path)
    base = os.path.splitext(csv_file)[0]
    wav_dir = csv_dir.replace("/csv", "/wav")
    return os.path.join(wav_dir, base) + "/"

def _cache_path(cache_root, csv_path, wav_filename):
    csv_dir = os.path.dirname(csv_path)
    group = os.path.basename(os.path.dirname(csv_dir))
    subset = os.path.splitext(os.path.basename(csv_path))[0]
    stem = os.path.splitext(os.path.basename(str(wav_filename).strip()))[0]
    return os.path.join(cache_root, group, subset, f"{stem}.npy")

def _get_cell(dfrow, names):
    for c in names:
        if c in dfrow.index and pd.notna(dfrow[c]):
            return str(dfrow[c]).strip()
    return None

def _chord_type_from_csv_name(csv_path: str) -> str:
    base = os.path.splitext(os.path.basename(csv_path))[0]
    # triads_major_loose → major, triads_diminished_strict → diminished
    parts = base.replace("triads_", "").split("_")
    return parts[0]  # major / minor / diminished / augmented

def load_triads_block(csv_list, domain_tag, cache_root):
    frames = []
    for path in csv_list:
        if not os.path.exists(path):
            print(f"   Warning: CSV not found, skipping: {path}")
            continue

        df = pd.read_csv(path)
        fname_col = resolve_filename_column(df)
        audio_root = derive_audio_root_from_csv(path)

        # Derive chord type from CSV filename
        chord_type = _chord_type_from_csv_name(path)

        # Standardise per-instrument dyn/tec columns
        std = {
            "dyn_cello": [], "tec_cello": [],
            "dyn_viola": [], "tec_viola": [],
            "dyn_violin1": [], "tec_violin1": [],
            "dyn_violin2": [], "tec_violin2": [],
        }

        for _, r in df.iterrows():
            d_global = _get_cell(r, ["dynamic"])
            t_global = _get_cell(r, ["technique"])

            for inst in INSTRUMENTS:
                d = _get_cell(r, _cand_dyn_cols(inst))
                t = _get_cell(r, _cand_tec_cols(inst))
                if d is None and d_global is not None:
                    d = d_global
                if t is None and t_global is not None:
                    t = t_global
                std[f"dyn_{inst}"].append(d)
                std[f"tec_{inst}"].append(t)

        df_std = pd.DataFrame(std)

        df["subset"] = domain_tag
        df["ensemble_size"] = "trio"
        df["voices"] = 3
        df["chord_type"] = chord_type

        # Build absolute paths
        df["filepath"] = df[fname_col].apply(
            lambda x: str(x).strip() if os.path.isabs(str(x).strip())
            else os.path.join(audio_root, str(x).strip())
        )
        df["cachefile"] = df[fname_col].apply(
            lambda x: _cache_path(CACHE_ROOT, path, x)
        )

        df = pd.concat([df.reset_index(drop=True), df_std.reset_index(drop=True)], axis=1)

        # Keep only rows where WAV exists
        df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)

        if len(df) == 0:
            print(f"   Warning: empty block after filtering for {path}")

        frames.append(df)

    return frames

all_dfs = []
all_dfs += load_triads_block(csv_trios_loose,  "loose",  CACHE_ROOT)
all_dfs += load_triads_block(csv_trios_strict,  "strict", CACHE_ROOT)

if not all_dfs:
    raise FileNotFoundError("No valid data was loaded.")

df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Loaded {len(df_all)} total samples")

print("\nChord-type distribution:")
print(df_all["chord_type"].value_counts())

print("\nSubset distribution:")
print(df_all["subset"].value_counts())

print("\nSubset × Chord type:")
print(pd.crosstab(df_all["subset"], df_all["chord_type"]))


## Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15)
Stratifies by chord_type × subset (loose/strict) to preserve label balance across splits. Saves indices to disk.


In [ ]:
# Cell 5 — Stratified Train / Val / Test Split (70 / 15 / 15)

OUT_SPLIT_DIR = Path(paths.split_dir)
OUT_SPLIT_DIR.mkdir(parents=True, exist_ok=True)

_split_file    = OUT_SPLIT_DIR / "exp2_split_indices.npz"
_manifest_file = OUT_SPLIT_DIR / "exp2_manifest.csv"

# ===================== CACHE VALIDATION =====================
use_cache = False

if _split_file.exists() and _manifest_file.exists():
    manifest = pd.read_csv(_manifest_file)
    if len(manifest) == len(df_all):
        use_cache = True
    else:
        print("Cached splits incompatible with current dataset — recomputing...")
        _split_file.unlink(missing_ok=True)
        _manifest_file.unlink(missing_ok=True)

# ===================== LOAD OR COMPUTE =====================
if use_cache:
    print("Loading existing splits from disk...")
    _npz      = np.load(_split_file)
    train_idx = _npz["train_idx"]
    val_idx   = _npz["val_idx"]
    test_idx  = _npz["test_idx"]

else:
    print("Computing splits...")

    # Stratify by chord_type × subset
    strat_key = df_all["chord_type"].astype(str) + "__" + df_all["subset"].astype(str)

    idx_all = np.arange(len(df_all))

    train_idx, tmp_idx = train_test_split(
        idx_all, test_size=0.30, random_state=SEED, stratify=strat_key)

    strat_tmp = strat_key.iloc[tmp_idx]
    val_idx, test_idx = train_test_split(
        tmp_idx, test_size=0.50, random_state=SEED, stratify=strat_tmp)

    # Verify disjointness
    S_tr, S_va, S_te = set(train_idx), set(val_idx), set(test_idx)
    assert len(S_tr & S_va) == 0
    assert len(S_tr & S_te) == 0
    assert len(S_va & S_te) == 0
    assert len(S_tr | S_va | S_te) == len(df_all)
    print("Disjointness & coverage: OK")

    # Save
    np.savez_compressed(_split_file,
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

    manifest = pd.DataFrame({
        "idx": np.arange(len(df_all)),
        "split": "none",
        "subset": df_all["subset"].values,
        "chord_type": df_all["chord_type"].values,
    })
    manifest.loc[train_idx, "split"] = "train"
    manifest.loc[val_idx,   "split"] = "val"
    manifest.loc[test_idx,  "split"] = "test"
    manifest.to_csv(_manifest_file, index=False)
    print("Saved splits")

total = len(train_idx) + len(val_idx) + len(test_idx)
print(f"\nSplit sizes — train: {len(train_idx)} ({len(train_idx)/total:.1%}) "
      f"| val: {len(val_idx)} ({len(val_idx)/total:.1%}) "
      f"| test: {len(test_idx)} ({len(test_idx)/total:.1%})")

# ===================== REPORT =====================
for name, idx in [("TRAIN", train_idx), ("VAL", val_idx), ("TEST", test_idx)]:
    print(f"\n{name} — chord_type × subset")
    print(pd.crosstab(df_all.iloc[idx]["subset"], df_all.iloc[idx]["chord_type"]))


## Cell 6b — Reload Splits from Disk
Run instead of Cell 5 on every session restart.

In [ ]:
# Cell 6b — Reload splits from disk

if "df_all" not in globals():
    raise RuntimeError("df_all is missing. Re-run Cells 3 → 4 first.")

OUT_SPLIT_DIR = Path(paths.split_dir)
NPZ_PATH      = OUT_SPLIT_DIR / "exp2_split_indices.npz"
CSV_PATH      = OUT_SPLIT_DIR / "exp2_manifest.csv"

for p in [NPZ_PATH, CSV_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Split file not found: {p}\nRun Cell 5 once to generate it.")

manifest = pd.read_csv(CSV_PATH)
if len(manifest) != len(df_all):
    raise RuntimeError("Split files are incompatible with current df_all. Re-run Cell 5.")

data      = np.load(NPZ_PATH)
train_idx = data["train_idx"]
val_idx   = data["val_idx"]
test_idx  = data["test_idx"]

df_train = df_all.iloc[train_idx].reset_index(drop=True)
df_val   = df_all.iloc[val_idx].reset_index(drop=True)
df_test  = df_all.iloc[test_idx].reset_index(drop=True)

total = len(train_idx) + len(val_idx) + len(test_idx)

print(f"Splits reloaded — "
      f"train: {len(df_train)} ({len(df_train)/total:.1%}) | "
      f"val: {len(df_val)} ({len(df_val)/total:.1%}) | "
      f"test: {len(df_test)} ({len(df_test)/total:.1%})")

s_tr = set(train_idx.tolist())
s_va = set(val_idx.tolist())
s_te = set(test_idx.tolist())

assert s_tr.isdisjoint(s_va)
assert s_tr.isdisjoint(s_te)
assert s_va.isdisjoint(s_te)

print("Disjointness: OK")

for name, df in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    print(f"\n{name} — chord_type × subset")
    print(pd.crosstab(df["subset"], df["chord_type"]))


## Cell 11.8 — Focus Chord-Type Selection (TRAIN only)
Defines `FOCUS` — the 4 triad chord types used for classification.

In [ ]:
# Cell 11.8 — Focus chord-type selection (EXP2)

if "df_train" not in globals():
    raise RuntimeError("df_train not found. Run Cells 3 → 4 → 5/6b first.")

FOCUS = ["major", "minor", "diminished", "augmented"]

observed = sorted(df_train["chord_type"].unique())
missing = [x for x in FOCUS if x not in observed]
if missing:
    raise RuntimeError(f"Focus chord types missing from training data: {missing}")

focus_counts = df_train["chord_type"].value_counts()
total = len(df_train)

print(f"FOCUS: {FOCUS}")
for ct in FOCUS:
    n = int(focus_counts.get(ct, 0))
    print(f"  {ct:15s}  {n:>8,}  ({100*n/total:.1f}%)")

os.makedirs(paths.split_dir, exist_ok=True)
with open(os.path.join(paths.split_dir, "focus_labels.json"), "w") as f:
    json.dump({"chord_types": FOCUS}, f, indent=2)

print("Saved →", os.path.join(paths.split_dir, "focus_labels.json"))


## Cell 11.10 — Class Weights for Imbalance Correction
Computes effective-number class weights for the 4 chord types.

In [ ]:
# Cell 11.10 — Class weights for imbalance (chord-type)

assert "FOCUS" in globals()
assert "df_train" in globals()

def _class_weights_from_counts(counts_dict, beta=0.9999):
    labels = list(counts_dict.keys())
    ns = np.array([max(1, counts_dict[k]) for k in labels], dtype=np.float64)
    w = (1.0 - beta) / (1.0 - np.power(beta, ns))
    w = w / w.mean()
    return labels, w.astype(np.float32)

chord_counts = {ct: int((df_train["chord_type"] == ct).sum()) for ct in FOCUS}
chord_keys, chord_w = _class_weights_from_counts(chord_counts, beta=0.9999)

assert chord_keys == FOCUS, "Weight order mismatch with FOCUS!"

CLASS_WEIGHTS = torch.tensor(chord_w, dtype=torch.float32)

print("Chord-type weights (mean≈1):", np.round(CLASS_WEIGHTS.numpy(), 3))
print("Counts:", chord_counts)

os.makedirs(paths.split_dir, exist_ok=True)
np.save(os.path.join(paths.split_dir, "class_weights_chord.npy"), CLASS_WEIGHTS.numpy())

with open(os.path.join(paths.split_dir, "class_weights_counts.json"), "w") as f:
    json.dump({"chord_counts": chord_counts}, f, indent=2)

print("Saved class weights →", paths.split_dir)


## Cell 11.11 — Dataset, Collate, and Split Construction (EXP2)
Defines `SECDTriadDataset` that returns `input_values` (128×256 mel) and
`labels` (single int chord-type label). This is a **global** classification
task — one label per sample, not per-instrument.

In [ ]:
# Cell 11.11 — Dataset, collate, split construction (EXP2)

for _n in ("df_train", "df_val", "df_test", "FOCUS", "INSTR_ORDER"):
    if _n not in globals():
        raise RuntimeError(f"{_n} missing — run previous cells.")

TARGET_FRAMES = 256

def pad_or_truncate(mel, n_frames):
    t = mel.shape[-1]
    if t >= n_frames:
        return mel[..., :n_frames]
    return torch.nn.functional.pad(mel, (0, n_frames - t))

chord2i = {s: i for i, s in enumerate(FOCUS)}

print("Chord-type classes:", FOCUS)
print("chord2i:", chord2i)

class SECDTriadDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            mel = torch.tensor(np.load(row["cachefile"]).astype(np.float32))
        except Exception:
            mel = torch.zeros(128, TARGET_FRAMES)

        mel = pad_or_truncate(mel, TARGET_FRAMES)

        label = chord2i.get(row["chord_type"], -1)

        return {
            "input_values": mel,
            "labels": torch.tensor(label, dtype=torch.long),
        }

def collate_fn(batch):
    return {
        "input_values": torch.stack([b["input_values"] for b in batch]),
        "labels": torch.stack([b["labels"] for b in batch]),
    }

ds_train = SECDTriadDataset(df_train)
ds_val   = SECDTriadDataset(df_val)
ds_test  = SECDTriadDataset(df_test)

print(f"Train: {len(ds_train)}")
print(f"Val  : {len(ds_val)}")
print(f"Test : {len(ds_test)}")

# Quick sanity check
sample = ds_train[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"input_values shape: {sample['input_values'].shape}")
print(f"label: {sample['labels'].item()} ({FOCUS[sample['labels'].item()]})")

b = collate_fn([ds_train[0], ds_train[1]])
print(f"\nBatch input shape: {b['input_values'].shape}")
print(f"Batch labels shape: {b['labels'].shape}")


## Cell 12 — Full Training Pipeline (EXP2 / triad chord-type classification)
Defines and trains `ASTMultiHeadTriadClassifier`: AST backbone with per-instrument
conditioning heads fused via a two-layer MLP for global chord-type prediction.
**This cell is fully self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.

In [ ]:
# Cell 12 — Full Training Pipeline (EXP2 / triad chord-type classification)

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ── Self-contained helpers ──────────────────────────────────

def resize_ast_pos_embed(ast_model, target_frames: int):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

# ── Bind upstream objects ──

CLASS_WEIGHTS_T = CLASS_WEIGHTS.detach().clone().float().cpu()
train_dataset = ds_train
val_dataset   = ds_val
test_dataset  = ds_test
data_collator = collate_fn

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found.")

print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(SEED)
np.random.seed(SEED)

stamp = time.strftime("%Y%m%d-%H%M%S")
OUTDIR = os.path.join(paths.outdir, f"exp2_ast_triads_full_{stamp}")
os.makedirs(OUTDIR, exist_ok=True)

with open(os.path.join(paths.outdir, "latest_run.txt"), "w", encoding="utf-8") as f:
    f.write(OUTDIR)

with open(os.path.join(OUTDIR, "session_config.json"), "w", encoding="utf-8") as f:
    json.dump({
        "TASK_NAME": TASK_NAME,
        "TASK_OBJECTIVE": TASK_OBJECTIVE,
        "MODEL_NAME": MODEL_NAME,
        "FOCUS": list(FOCUS),
        "TARGET_FRAMES": int(TARGET_FRAMES),
        "INSTR_ORDER": list(INSTR_ORDER),
        "CLASS_WEIGHTS": CLASS_WEIGHTS_T.tolist(),
    }, f, indent=2)

# ── Metrics ──

def compute_metrics_exp2(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    preds = logits.argmax(axis=-1).reshape(-1)
    labels = labels.reshape(-1)

    mask = labels >= 0
    yt = labels[mask]
    yp = preds[mask]

    return {
        "eval_support": int(mask.sum()),
        "eval_acc": float(accuracy_score(yt, yp)),
        "eval_f1_macro": float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

# ── Model ──

class ASTMultiHeadTriadClassifier(nn.Module):
    def __init__(self, model_name, instr_order, n_classes, class_weights):
        super().__init__()
        self.n_instr = len(instr_order)
        self.n_classes = n_classes

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.inst_embed = nn.Embedding(self.n_instr, d)

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Dropout(0.10),
                nn.Linear(d, d),
                nn.GELU(),
            )
            for _ in range(self.n_instr)
        ])

        self.fusion = nn.Sequential(
            nn.Linear(self.n_instr * d, d),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(d, n_classes),
        )

        if isinstance(class_weights, torch.Tensor):
            cw = class_weights.detach().clone().float()
        else:
            cw = torch.tensor(class_weights, dtype=torch.float32)
        self.register_buffer("w_cls", cw)

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, labels=None, **kwargs):
        emb = self.feats(input_values)

        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)

        conditioned_feats = []
        for i, head in enumerate(self.heads):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            conditioned = emb + inst_vec
            conditioned_feats.append(head(conditioned))

        fused = torch.cat(conditioned_feats, dim=-1)
        logits = self.fusion(fused)

        loss = None
        if labels is not None:
            loss = masked_ce(logits, labels, weight=self.w_cls)

        return {"loss": loss, "logits": logits}

# ── Live progress callback ──

class EpochTableCallback(TrainerCallback):
    def __init__(self):
        self.train_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        if state.epoch == 1:
            print("-" * 84)
            print("Epoch | Train Loss | Val Loss | Acc    | F1")
            print("-" * 84)

        ep = int(state.epoch or 0)
        tl = float(np.mean(self.train_losses)) if self.train_losses else 0.0
        self.train_losses = []

        vl  = float(metrics.get("eval_loss", 0.0))
        acc = float(metrics.get("eval_acc", 0.0))
        f1  = float(metrics.get("eval_f1_macro", 0.0))

        print(f"{ep:>5} | {tl:.4f}     | {vl:.4f}   | {acc:.4f} | {f1:.4f}")

# ── Build & launch ──

model = ASTMultiHeadTriadClassifier(
    MODEL_NAME,
    INSTR_ORDER,
    len(FOCUS),
    CLASS_WEIGHTS_T,
).to(torch.device("cuda:0"))

model.ast.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=os.path.join(OUTDIR, "hf_ckpt"),
    logging_dir=os.path.join(OUTDIR, "logs"),

    num_train_epochs=40,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,

    learning_rate=3e-5,
    warmup_ratio=0.05,
    weight_decay=1e-2,
    fp16=True,
    dataloader_num_workers=4,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=2,

    remove_unused_columns=False,
    label_names=["labels"],
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics_exp2,
    callbacks=[
        EpochTableCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001,
        ),
    ],
)

trainer.remove_callback(PrinterCallback)

print("-" * 72)
print(f"Run  : {OUTDIR}")
print("-" * 72)

t0 = time.time()
trainer.train()

print(f"\nTraining finished in {(time.time() - t0) / 60:.2f} min")

trainer.save_model(os.path.join(args.output_dir, "final"))
trainer.save_state()

print("-" * 72)
print("FULL RUN COMPLETE")
print(f"Artifacts -> {OUTDIR}")
print("-" * 72)


## Cell 13 — Post-Training Evaluation and Artifacts (EXP2 global triad task)
Loads the best checkpoint and runs inference on val/test sets.
**Self-contained** — `resize_ast_pos_embed` and `masked_ce` are defined inline.

In [ ]:
# Cell 13 — Post-Training Evaluation and Artifacts (EXP2 global triad task)

matplotlib.use("Agg")

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Self-contained helpers ──────────────────────────────────

def resize_ast_pos_embed(ast_model, target_frames: int):
    n_mels_patches = (128 - 16) // 10 + 1
    n_time_patches = (target_frames - 16) // 10 + 1
    n_patches = n_mels_patches * n_time_patches
    n_tokens = n_patches + 2
    pos_embed = ast_model.embeddings.position_embeddings
    old_n = pos_embed.shape[1]
    if old_n == n_tokens:
        return
    cls_dist = pos_embed[:, :2, :]
    patch_pos = pos_embed[:, 2:, :]
    old_freq, old_time, d = 12, 101, patch_pos.shape[-1]
    patch_pos = patch_pos.reshape(1, old_freq, old_time, d).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(
        patch_pos.float(),
        size=(n_mels_patches, n_time_patches),
        mode="bicubic",
        align_corners=False,
    )
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, n_patches, d)
    new_pos = torch.cat([cls_dist, patch_pos], dim=1)
    ast_model.embeddings.position_embeddings = nn.Parameter(new_pos)

def masked_ce(logits, targets, weight=None, ignore_index=-1):
    valid = targets.ne(ignore_index)
    if not valid.any():
        return logits.new_zeros(())
    return F.cross_entropy(
        logits[valid].float(),
        targets[valid],
        weight=weight.float() if weight is not None else None,
        label_smoothing=0.05,
    )

CLASS_WEIGHTS_T = CLASS_WEIGHTS.detach().clone().float().cpu()

if "OUTDIR" not in globals() or not os.path.isdir(globals().get("OUTDIR", "")):
    ptr = os.path.join(paths.outdir, "latest_run.txt")
    if not os.path.exists(ptr):
        raise RuntimeError("OUTDIR not found and latest_run.txt is missing.")
    with open(ptr, "r", encoding="utf-8") as f:
        OUTDIR = f.read().strip()
    print(f"Loaded run path -> {OUTDIR}")
else:
    print(f"Using OUTDIR -> {OUTDIR}")

CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")
ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

cfg_path = os.path.join(OUTDIR, "session_config.json")
if not os.path.exists(cfg_path):
    raise RuntimeError("Missing session_config.json")

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg = json.load(f)

FOCUS = list(cfg["FOCUS"])
TARGET_FRAMES = int(cfg["TARGET_FRAMES"])
INSTR_ORDER = list(cfg["INSTR_ORDER"])
MODEL_NAME = cfg["MODEL_NAME"]
W_CLS = np.array(cfg["CLASS_WEIGHTS"], dtype=np.float32)
N_CLASSES = len(FOCUS)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device -> {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

need = ["val_dataset", "test_dataset", "data_collator"]
# FIX: renamed loop variable from 'n' to 'req' to avoid shadowing numpy's 'n' alias
for req in need:
    if req not in globals():
        raise RuntimeError(f"Missing: {req}")

print(f"VAL: {len(val_dataset):,} | TEST: {len(test_dataset):,}")

class ASTMultiHeadTriadClassifier(nn.Module):
    def __init__(self, model_name, instr_order, n_classes, class_weights):
        super().__init__()
        self.n_instr = len(instr_order)
        self.n_classes = n_classes

        self.ast = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        resize_ast_pos_embed(self.ast, TARGET_FRAMES)

        with torch.no_grad():
            dummy = torch.zeros(1, 128, TARGET_FRAMES)
            h = self.ast(input_values=dummy).last_hidden_state
            d = h.mean(dim=1).shape[-1]

        self.inst_embed = nn.Embedding(self.n_instr, d)
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d, d),
                nn.GELU(),
                nn.Dropout(0.10),
                nn.Linear(d, d),
                nn.GELU(),
            )
            for _ in range(self.n_instr)
        ])
        self.fusion = nn.Sequential(
            nn.Linear(self.n_instr * d, d),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(d, n_classes),
        )
        self.register_buffer("w_cls", torch.tensor(class_weights, dtype=torch.float32))

    def feats(self, x):
        return self.ast(input_values=x).last_hidden_state.mean(dim=1)

    def forward(self, input_values, labels=None, **kwargs):
        emb = self.feats(input_values)
        inst_ids = torch.arange(self.n_instr, device=emb.device)
        inst_vecs = self.inst_embed(inst_ids)

        conditioned_feats = []
        for i, head in enumerate(self.heads):
            inst_vec = inst_vecs[i].unsqueeze(0).expand_as(emb)
            conditioned = emb + inst_vec
            conditioned_feats.append(head(conditioned))

        fused = torch.cat(conditioned_feats, dim=-1)
        logits = self.fusion(fused)

        loss = None
        if labels is not None:
            loss = masked_ce(logits, labels, weight=self.w_cls)
        return {"loss": loss, "logits": logits}

def resolve_best_checkpoint(ckpt_dir):
    state_path = os.path.join(ckpt_dir, "trainer_state.json")
    if not os.path.exists(state_path):
        return None
    with open(state_path, "r", encoding="utf-8") as f:
        state = json.load(f)
    best = state.get("best_model_checkpoint")
    if best and os.path.isdir(best):
        print(f"BEST checkpoint -> {best}")
        return best
    return None

model_dir = resolve_best_checkpoint(CKPT_DIR)

if model_dir is None:
    checkpoints = sorted(
        glob.glob(os.path.join(CKPT_DIR, "checkpoint-*")),
        key=lambda x: int(x.split("-")[-1])
    )
    if not checkpoints:
        raise RuntimeError("No checkpoints found")
    model_dir = checkpoints[-1]
    print(f"Fallback -> {model_dir}")

model = ASTMultiHeadTriadClassifier(MODEL_NAME, INSTR_ORDER, N_CLASSES, W_CLS)

state_dict = st.load_file(
    os.path.join(model_dir, "model.safetensors"),
    device=str(device)
)
load_info = model.load_state_dict(state_dict, strict=False)
if load_info.missing_keys:
    print(f"Missing keys    : {load_info.missing_keys}")
if load_info.unexpected_keys:
    print(f"Unexpected keys : {load_info.unexpected_keys}")

model.to(device)
model.eval()

eval_trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=ARTIFACT_DIR,
        per_device_eval_batch_size=64,
        report_to=[],
        remove_unused_columns=False,
        label_names=["labels"],
        use_cpu=(device.type == "cpu"),
        seed=SEED,
    ),
    data_collator=data_collator,
)

def run_predictions(dataset):
    pred = eval_trainer.predict(dataset)
    raw_logits = pred.predictions
    raw_labels = pred.label_ids

    logits = np.asarray(raw_logits[0]) if isinstance(raw_logits, tuple) else np.asarray(raw_logits)
    labels = np.asarray(raw_labels[0]) if isinstance(raw_labels, tuple) else np.asarray(raw_labels)

    pred_ids = logits.argmax(axis=-1).reshape(-1)
    labels = labels.reshape(-1)

    mask = labels >= 0
    y_true = labels[mask]
    y_pred = pred_ids[mask]

    return y_true, y_pred, logits

def save_cm(y_true, y_pred, labels, title, path, normalize=None):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
        normalize=normalize
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    fmt = ".2f" if normalize is not None else "d"
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format=fmt
    )
    ax.set_title(title, fontsize=12, pad=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

all_metrics = {}

for split_name, dataset in [("val", val_dataset), ("test", test_dataset)]:
    print(f"\n=== {split_name.upper()} ===")

    y_true, y_pred, logits = run_predictions(dataset)

    rep = classification_report(
        y_true,
        y_pred,
        target_names=FOCUS,
        output_dict=True,
        zero_division=0,
    )

    all_metrics[split_name] = rep

    print(f"{split_name} | acc={rep['accuracy']:.4f} | f1={rep['macro avg']['f1-score']:.4f}")

    save_cm(
        y_true, y_pred, FOCUS,
        f"{split_name.upper()} | Triad chord type — Absolute",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_abs.png"),
        normalize=None,
    )
    save_cm(
        y_true, y_pred, FOCUS,
        f"{split_name.upper()} | Triad chord type — Normalized",
        os.path.join(ARTIFACT_DIR, f"cm_{split_name}_norm.png"),
        normalize="true",
    )

with open(os.path.join(ARTIFACT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2)

pngs = sorted(f for f in os.listdir(ARTIFACT_DIR) if f.endswith(".png"))
print(f"\nArtifacts saved -> {ARTIFACT_DIR}")
print(f"metrics.json + {len(pngs)} PNGs")
for p in pngs:
    print(f"  {p}")


## Cell 13.05 — Training Curves (Loss + F1)

Reads `trainer_state.json` from the HF checkpoint directory and parses `log_history` to extract per-epoch train loss, validation loss, validation accuracy, and validation macro F1. Produces two PNGs saved to `OUTDIR/artifacts/`: `loss_curves.png` (train vs val loss) and `f1_curves.png` (val accuracy + macro F1 over epochs). The F1 plot is skipped gracefully if the log history contains neither `eval_acc` nor `eval_f1_macro` entries.

In [ ]:
# Cell 13.05 — Training Curves (Loss + F1)

matplotlib.use("Agg")

ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")
CKPT_DIR = os.path.join(OUTDIR, "hf_ckpt")

state_path = os.path.join(CKPT_DIR, "trainer_state.json")
if not os.path.exists(state_path):
    raise RuntimeError(f"trainer_state.json not found in {CKPT_DIR}")

with open(state_path, "r", encoding="utf-8") as f:
    state = json.load(f)

log = state["log_history"]

train_epochs, train_loss = [], []
val_epochs, val_loss, val_acc, val_f1 = [], [], [], []

for entry in log:
    e = entry.get("epoch")
    if e is None:
        continue
    if "loss" in entry and "eval_loss" not in entry:
        train_epochs.append(e)
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        val_epochs.append(e)
        val_loss.append(entry["eval_loss"])
        val_acc.append(entry.get("eval_acc", None))
        val_f1.append(entry.get("eval_f1_macro", None))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_epochs, train_loss, "o-", ms=3, label="Train loss")
ax.plot(val_epochs, val_loss, "s-", ms=3, label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, "loss_curves.png"), dpi=150)
plt.close(fig)
print("loss_curves.png")

has_acc = any(v is not None for v in val_acc)
has_f1  = any(v is not None for v in val_f1)

if has_acc or has_f1:
    fig, ax = plt.subplots(figsize=(9, 5))
    if has_acc:
        ax.plot(val_epochs, val_acc, "s-", ms=3, label="Val accuracy")
    if has_f1:
        ax.plot(val_epochs, val_f1, "^-", ms=3, label="Val macro F1")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_title("Validation Accuracy & Macro F1")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(ARTIFACT_DIR, "f1_curves.png"), dpi=150)
    plt.close(fig)
    print("f1_curves.png")
else:
    print("No eval_acc / eval_f1_macro found — f1_curves.png skipped")

print(f"Saved -> {ARTIFACT_DIR}")


## Cell 13.08 — Load and Print Classification Reports from metrics.json

In [ ]:
# Cell 13.08 — Load and Print Classification Reports from metrics.json

json_path = os.path.join(ARTIFACT_DIR, "metrics.json")

if not os.path.exists(json_path):
    print("❌ metrics.json not found!")
else:
    with open(json_path, "r", encoding="utf-8") as f:
        all_metrics = json.load(f)

    print("="*65)
    print("CLASSIFICATION REPORTS FROM SAVED METRICS.JSON")
    print("="*65)

    for split in ["val", "test"]:
        if split not in all_metrics:
            continue

        data = all_metrics[split]
        print(f"\n\n=== {split.upper()} SET CLASSIFICATION REPORT ===\n")

        # Pretty table
        print(f"{'Class':<12} {'Precision':>9} {'Recall':>9} {'F1-Score':>9} {'Support':>8}")
        print("-" * 65)

        for cls in FOCUS:
            m = data[cls]
            print(f"{cls:<12} {m['precision']:9.4f} {m['recall']:9.4f} {m['f1-score']:9.4f} {int(m['support']):8d}")

        # Macro / Weighted / Accuracy
        macro = data['macro avg']
        acc = data['accuracy']

        print("-" * 65)
        print(f"{'Accuracy':<12} {acc:9.4f}")
        print(f"{'Macro Avg':<12} {macro['precision']:9.4f} {macro['recall']:9.4f} {macro['f1-score']:9.4f}")
        print("="*65)


## Cell 13.1 — Artifact Preview (EXP2 global triad task)

Displays all PNG artifacts produced by Cells 13 and 13.05 inline in the notebook using `IPython.display`. The images are shown in a fixed order: loss curves, F1 curves, then confusion matrices (val absolute → val normalised → test absolute → test normalised). Missing files are listed at the end rather than raising an error, so the cell can be run even if only a subset of plots was generated.

In [ ]:
# Cell 13.1 — Artifact Preview (EXP2 global triad task)

ARTIFACT_DIR = os.path.join(OUTDIR, "artifacts")

if not os.path.isdir(ARTIFACT_DIR):
    raise RuntimeError(f"Artifacts directory not found: {ARTIFACT_DIR}")

PNG_ORDER = [
    "loss_curves.png",
    "f1_curves.png",
    "cm_val_abs.png",
    "cm_val_norm.png",
    "cm_test_abs.png",
    "cm_test_norm.png",
]

print("\n" + "=" * 70)
print("  Artifact Preview — EXP2 Triad Chord-Type Classification")
print("=" * 70)

missing = []

for fname in PNG_ORDER:
    fpath = os.path.join(ARTIFACT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"\n-- {fname} --")
        display(IPImage(filename=fpath))
    else:
        missing.append(fname)

if missing:
    print("\n" + "-" * 70)
    print("Missing artifacts:")
    for m in missing:
        print(f"  {m}")
